In [1]:
import sys
sys.path.insert(1, '../../../Utils/')
from unet_utils import *
sys.path.insert(1, '../../../Utils/')
from metrics import *
from pathlib import Path
import cv2
import imutils
from sklearn.model_selection import train_test_split
from sklearn.utils import shuffle
from sklearn.model_selection import KFold
import matplotlib.pyplot as plt

In [2]:
tf.config.list_physical_devices('GPU')

[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]

# Dataset 

In [3]:
path=Path('../../../../Datasets/Processed/dataset/')
path_tot=path/'train_def'
path_test=path/'test_def'
path_gt=path/'GT_ICM'
path_models=Path('Harun/ICM')

In [4]:
dim=(256,256)
def load_imgs(path, path_gt=''):
    files=[path/f for f in os.listdir(path)]
    x = np.array([cv2.resize(cv2.imread(str(f),cv2.IMREAD_GRAYSCALE), dim) for f in files])
    if path_gt == '':
        y=None
    else:
        y = np.array([cv2.threshold(cv2.resize(cv2.imread(str(path_gt/(f.stem+' ICM_Mask.bmp')),cv2.IMREAD_GRAYSCALE), dim),127,255,cv2.THRESH_BINARY)[1] for f in files])
    return x,y

In [5]:
dim=(256,256)
x,y=load_imgs(path_tot,path_gt)

In [6]:
def std_norm(x):
    mean, std= np.mean(x, axis=0), np.std(x, axis=0) 
    return ((x.astype('float32') - mean) / std , mean, std)

def std_norm_test(x,mean,std):
    return (x.astype('float32') - mean) / std

In [7]:
kfold = KFold(n_splits=10, shuffle=True, random_state=42)

# Model

In [8]:
model= build_unet(input_shape=(256,256,1))

In [9]:
callbacks = [
    EarlyStopping(patience=15),
    ReduceLROnPlateau(factor=0.05, patience=5)]

In [11]:
fold_no = 1
for train, test in kfold.split(x):
    if fold_no>0:
        x_train, y_train = x[train], y[train]
        x_test, y_test = x[test], y[test]

        x_train, mean, std = std_norm(x_train)
        x_test = std_norm_test(x_test, mean, std)

        x_train, y_train = data_augmentation(x_train, y_train)
        x_train, y_train = shuffle(x_train, y_train, random_state=13)

        x_train = x_train.astype("float32") 
        y_train = y_train.astype("float32")/255  

        x_test = x_test.astype("float32") 
        y_test = y_test.astype("float32")/255  

        train_gen = DataGenerator(x_train, y_train, 16)
        model= build_unet(input_shape=(256,256,1))
        model.compile(optimizer=Adam(learning_rate=0.0001), loss=my_loss_fn, metrics=[jaccard_index])

        print(f'Training for fold {fold_no} ...')
        history = model.fit(train_gen,batch_size=16, epochs=200, callbacks=callbacks, validation_data=(x_test, y_test))
        predictions = model.predict(x_test)


        model.save(path_models/('fold'+str(fold_no)))   
    fold_no = fold_no + 1

Training for fold 1 ...
Epoch 1/200
438/438 [==============================] - 141s 251ms/step - loss: 1.5413 - jaccard_index: 0.4794 - val_loss: 1.3186 - val_jaccard_index: 0.5044
Epoch 2/200
438/438 [==============================] - 106s 242ms/step - loss: 1.0866 - jaccard_index: 0.6666 - val_loss: 1.0540 - val_jaccard_index: 0.6555
Epoch 3/200
438/438 [==============================] - 106s 241ms/step - loss: 1.0253 - jaccard_index: 0.7065 - val_loss: 1.0236 - val_jaccard_index: 0.6871
Epoch 4/200
438/438 [==============================] - 106s 241ms/step - loss: 0.9905 - jaccard_index: 0.7305 - val_loss: 1.0148 - val_jaccard_index: 0.6865
Epoch 5/200
438/438 [==============================] - 106s 241ms/step - loss: 0.9669 - jaccard_index: 0.7472 - val_loss: 1.0611 - val_jaccard_index: 0.6579
Epoch 6/200
438/438 [==============================] - 106s 241ms/step - loss: 0.9506 - jaccard_index: 0.7591 - val_loss: 1.0348 - val_jaccard_index: 0.6785
Epoch 7/200
438/438 [=============

# Metrics

In [8]:
x_test, y_test = load_imgs(path_test,path_gt)
_, mean, std = std_norm(x)
x_test = std_norm_test(x_test, mean, std) 
x_test = x_test.astype("float32") 
y_test = y_test.astype("float32")/255 

In [9]:
def accuracy(target, prediction):
    true_detec = np.logical_not(np.logical_xor(target, prediction))
    return np.sum(true_detec)/np.sum(np.ones_like(target))

def precision(target, prediction):
    intersection = np.logical_and(target, prediction)
    return np.sum(intersection)/(np.sum(prediction)+1)

def recall(target, prediction):
    intersection = np.logical_and(target, prediction)
    return np.sum(intersection)/np.sum(target)

def jaccard(target, prediction):
    intersection = np.logical_and(target, prediction)
    union = np.logical_or(target, prediction)
    return np.sum(intersection) / np.sum(union)

def dice(target, prediction):
    intersection = np.logical_and(target, prediction)
    return 2*np.sum(intersection) / (np.sum(target) + np.sum(prediction))

def metrics(target, prediction):
    return {'accuracy': accuracy(target, prediction),
            'precision': precision(target, prediction),
            'recall': recall(target, prediction),
            'specificity': recall(1-target,1- prediction),
            'jaccard':jaccard(target, prediction),
            'dice': dice(target, prediction)}

def summary_metrics(y_test,predictions,thresh=0.5):
    a,p,r,s,j,d=0.,0.,0.,0.,0.,0.
    n=len(predictions)
    for i in range(n):
        preds= (predictions[i][:,:,0]>=thresh).astype('uint8')
        gt= y_test[i].astype('uint8')
        metricas=metrics(gt,preds)
        a+=metricas['accuracy']
        p+=metricas['precision']
        r+=metricas['recall']
        s+=metricas['specificity']
        j+=metricas['jaccard']
        d+=metricas['dice']
    return {'accuracy': a/n,
            'precision': p/n,
            'recall': r/n,
            'specificity': s/n,
            'jaccard':j/n,
            'dice': d/n}

In [10]:
m1=[]
for fold_no in range(1,11): 
    model= tf.keras.models.load_model(path_models/'fold{}'.format(fold_no),custom_objects={'jaccard_index':jaccard_index, 'my_loss_fn':my_loss_fn})
    predictions = model.predict(x_test)
    m1.append(summary_metrics(y_test,predictions, 0.5))

In [11]:
accuracy=[el['accuracy'] for el in m1]
precision=[el['precision'] for el in m1]
recall=[el['recall'] for el in m1]
specificity=[el['specificity'] for el in m1]
jaccard=[el['jaccard'] for el in m1]
dice=[el['dice'] for el in m1]

In [13]:
np.mean(accuracy), np.std(accuracy)

(0.9678273251182155, 0.004380214179414501)

In [14]:
np.mean(precision), np.std(precision)

(0.8513521295456213, 0.0283997079190889)

In [15]:
np.mean(recall), np.std(recall)

(0.66719869234082, 0.078315976627708)

In [16]:
np.mean(specificity), np.std(specificity)

(0.9943557489462453, 0.0023977334376856247)

In [17]:
np.mean(jaccard), np.std(jaccard)

(0.6101170925914003, 0.05844146766509681)

In [18]:
np.mean(dice), np.std(dice)

(0.7141809476528952, 0.06005842484243083)

In [22]:
for m in m1:
    print('accuracy: ' +str(m['accuracy']))
    print('precision: ' +str(m['precision']))
    print('recall: ' +str(m['recall']))
    print('jaccard: ' +str(m['jaccard']))
    print('--------------------------')

accuracy: 0.9677553678813734
precision: 0.8766313887615065
recall: 0.6380064764033003
jaccard: 0.5961578436178032
--------------------------
accuracy: 0.9755923622532895
precision: 0.8748875354204666
recall: 0.8142904060886341
jaccard: 0.7178581036894287
--------------------------
accuracy: 0.9718266537314967
precision: 0.8416932019219222
recall: 0.7079249430237368
jaccard: 0.6443589586319366
--------------------------
accuracy: 0.9686636673776727
precision: 0.8621086387590616
recall: 0.692734478394633
jaccard: 0.628170623683738
--------------------------
accuracy: 0.9622991461502878
precision: 0.8176529069671525
recall: 0.5774047021846587
jaccard: 0.54145055097175
--------------------------
accuracy: 0.9678983186420641
precision: 0.8618515179800486
recall: 0.6446764826395449
jaccard: 0.6003281961504005
--------------------------
accuracy: 0.9703288831208882
precision: 0.8898974090200236
recall: 0.7473821133938691
jaccard: 0.6621681933303631
--------------------------
accuracy: 0.96642

In [32]:
m1